# Frozen availability-shift study: reproduction

The supplied paper already reports the locked 2025 evaluation. Rerunning this notebook reproduces an existing experiment; it is not a fresh confirmatory test. Data are actual public CDC records. Each code cell performs one task. The notebook has been syntax-checked; Colab/Drive integration requires execution in your account.


In [ ]:
from google.colab import files
uploaded = files.upload()  # Choose the delivered research package ZIP.


In [ ]:
from pathlib import Path
import zipfile, os
archive = next(name for name in uploaded if name.endswith('.zip'))
project = Path('/content/brfss_project')
project.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for name in z.namelist():
        if not (project / name).resolve().is_relative_to(project.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(project)
os.chdir(project)
print(Path.cwd())


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch==2.14.1', '--index-url', 'https://download.pytorch.org/whl/cpu'], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-availability.txt'], check=True)


If Colab asks for a runtime restart after dependency installation, restart it and restore the working directory before continuing.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/BRFSS')
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = Path('/content/drive/MyDrive/BRFSS_reproduction')
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(DATA_DIR, RUN_DIR)


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, 'scripts/download_brfss.py', '--directory', str(DATA_DIR)], check=True)


In [ ]:
import os
for name in ['OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS']:
    os.environ[name] = '2'
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_availability_study.py', '-v'], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'truststroke.run_availability_study', 'fit', '--config', 'configs/availability_study_frozen.json', '--data', str(DATA_DIR / 'LLCP2024.XPT'), '--output', str(RUN_DIR / 'fit')], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'truststroke.run_availability_study', 'evaluate', '--run', str(RUN_DIR / 'fit'), '--data', str(DATA_DIR / 'LLCP2025.XPT'), '--output', str(RUN_DIR / 'evaluation2025')], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'truststroke.package_availability', '--run', str(RUN_DIR / 'fit'), '--external', str(RUN_DIR / 'evaluation2025'), '--development-data', str(DATA_DIR / 'LLCP2024.XPT'), '--output', str(RUN_DIR / 'aggregates')], check=True)


In [ ]:
import json
result = json.loads((RUN_DIR / 'evaluation2025' / 'results.json').read_text())
print(json.dumps(result['primary'], indent=2))


Do not tune the model against the 2025 results and call a later rerun an independent validation. Output directories are never overwritten. The paper describes stroke history, not future stroke incidence or clinical diagnosis.


## Reconstructed post-review analyses (v3)
These reproduce existing post-hoc evidence, not fresh confirmation on2025. The final cell fits18 extra classifiers.


In [ ]:
AUDIT_DIR = RUN_DIR / 'revision_audit'
subprocess.run([sys.executable, '-m', 'truststroke.revision_audit', 'diagnostics', '--data', str(DATA_DIR / 'LLCP2025.XPT'), '--run', str(RUN_DIR / 'fit'), '--external', str(RUN_DIR / 'evaluation2025'), '--output', str(AUDIT_DIR)], check=True)


In [ ]:
subprocess.run(['apt-get', 'install', '-y', 'r-base-core', 'r-cran-survey'], check=True)


In [ ]:
subprocess.run(['Rscript', 'scripts/verify_survey_domain.R', str(RUN_DIR / 'evaluation2025' / 'posthoc_survey_domain.csv'), str(AUDIT_DIR / 'r_survey.csv')], check=True)


In [ ]:
subprocess.run([sys.executable, '-m', 'truststroke.revision_audit', 'robustness', '--data', str(DATA_DIR / 'LLCP2024.XPT'), '--data25', str(DATA_DIR / 'LLCP2025.XPT'), '--run', str(RUN_DIR / 'fit'), '--external', str(RUN_DIR / 'evaluation2025'), '--output', str(AUDIT_DIR), '--robustness-output', str(RUN_DIR / 'posthoc-fitting-samples')], check=True)
